In [59]:
from pathlib import Path
from itertools import islice
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

In [66]:
RAPPORTS = Path("../rapports")


In [67]:
loader = PyMuPDF4LLMLoader(RAPPORTS / "bnpparibas_2025.pdf", mode="page")

In [75]:
next(loader.lazy_load()).page_content

'Ce document d’enregistrement universel est une reproduction de la version officielle du document d’enregistrement universel incluant le rapport financier annuel 2025 qui a été établie au format XHTML et est disponible sur notre site internet https://invest.bnpparibas \n\n'

In [76]:
liste=[]
for page in loader.lazy_load():
    liste.append(page.page_content)

In [62]:
BANQUES = {
    "bnpparibas" : ('BNP Paribas', 'fr'),
    "ca": ('Credit Agricole', 'fr'),
    "ubs": ('UBS','en')
}

In [63]:
"""
Prend en entree le path d'un pdf ( dans format "banque_annee.pdf") et retourne une liste composee des pages du pdf en format 
Document dont les metadata sont adaptees au format voulu
"""
def load_report(path):
    cle, annee = path.stem.rsplit('_',1)
    banque, langue = BANQUES[cle][0],BANQUES[cle][1]
    loader = PyMuPDF4LLMLoader(path,mode = "page")
    liste_page = []
    for p in loader.lazy_load():
        p.metadata = {
            "source": path.name,
            "banque": banque,
            "annee": int(annee),
            "langue": langue, 
            "page": p.metadata['page']+1
        }
        liste_page.append(p)
    return liste_page    

In [12]:
test_ca = load_reports(rapports / "ca_2025.pdf")

In [17]:
test_ca[50].metadata

{'source': 'ca_2025.pdf',
 'banque': 'Credit Agricole',
 'annee': '2025',
 'langue': 'fr',
 'page': 51}

In [64]:
"""
Returne liste composee de toutes les pages de tous les rapports du dossier ../rapports
"""
def load_reports(root_path):
    all_pages = []
    for pdf in sorted(root_path.glob('*.pdf')):
        pages = load_report(pdf)
        all_pages.extend(pages)
        print(f'{len(pages)} pages du document {pdf.name} ont ete lues.')
    return all_pages

In [65]:
docs = load_reports(rapports)

936 pages du document bnpparibas_2025.pdf ont ete lues.
272 pages du document ca_2025.pdf ont ete lues.
380 pages du document ubs_2025.pdf ont ete lues.


In [43]:
vide = [d for d in docs if len(d.page_content.strip())<50]

In [47]:
vide[1].metadata

{'source': 'ubs_2025.pdf',
 'banque': 'UBS',
 'annee': '2025',
 'langue': 'en',
 'page': 380}

In [58]:
docs[637].page_content

'**Updated assessment of proposed changes to the Swiss capital framework** \n\n_CET1, USD bn; % is CET1 capital ratio_ \n\n\n\n<!-- Start of picture text -->\nUBS AG, standalone UBS Group AG, consolidated<br>4Q25 4Q25<br>Incremental capital at UBS AG  22 91<br>to maintain 12.5% CET1 ratio<br>18.5%<br>causes material overshooting at  77<br>the Group consolidated level 69¹ (14) 16.5%<br>65¹<br>14.0%<br>12.5% (2)<br>(3)<br>22 44<br>12.5%<br>21<br>4Q25:  4Q25:<br>74bn /  (39)³ 6.1% 71bn /<br>14.2% 14.4%<br>Lower  Expected  Deduction  Full  Pro-forma  Gap to  Lower  ~14%  Capital  Pro-forma Deduction  Pro-forma<br>end of  repatriations  of TD DTAs,  deduction  post  guidance end of  CET1 ratio  build-up at  of TD DTAs,  post<br>12.5 –13.0%  in 2026² software  of foreign  repatriations  12.5 –13.0%  guidance UBS AG software and  deduction<br>CET1 ratio  and PVAs participations and current  CET1 ratio  PVAs of TD DTAs,<br>guidance proposal 5 guidance software<br>and PVAs<br>RWA, USD bn<br>523